# El problema de les N reines

Col·loca una reina per columna de manera que cap parella compartisca fila o diagonal. L'estat `(1, 3, 0, 2)` indica la fila de cada reina, d'esquerra a dreta. Primer col·locarem les reines una a una. Després començarem amb totes col·locades i intentarem reduir els conflictes.

Compara aquests dos taulers de quatre reines. En el primer, `(0, 1, 2, 3)`, totes estan en la mateixa diagonal. El segon, `(1, 3, 0, 2)`, és vàlid.

```text
Q...    ..Q.
.Q..    Q...
..Q.    ...Q
...Q    .Q..
```

In [ ]:
from random import Random

def xoquen(a, fila_a, b, fila_b):
    return fila_a == fila_b or abs(a - b) == abs(fila_a - fila_b)

def mostra_tauler(estat):
    for fila in range(len(estat)):
        print(" ".join("♛" if f == fila else "·" for f in estat))

## Backtracking
Provem una fila per a la columna següent. Si entra en conflicte amb alguna reina ja col·locada, la descartem. Si cap fila porta a una solució, tornem a la columna anterior.

In [ ]:
def backtracking(n, estat=()):
    columna = len(estat)
    if columna == n:
        return estat

    for fila in range(n):
        if any(xoquen(i, f, columna, fila) for i, f in enumerate(estat)):
            continue
        solucio = backtracking(n, estat + (fila,))
        if solucio is not None:
            return solucio
    return None

mostra_tauler(backtracking(8))

## Min-conflicts

A partir de 18 reines, el temps d'execució és molt alt perquè l'espai de cerca creix ràpidament. El nombre conegut de disposicions vàlides és de 666.090.624 per a 18 reines i de 2.691.008.701.644 per a 22 reines ([seqüència A000170 de l'OEIS](https://oeis.org/A000170)). Aquestes xifres compten solucions vàlides; no indiquen quants estats visitarà el nostre algorisme de backtracking abans de trobar-ne una.

Comencem amb una fila aleatòria per columna. A cada pas triem una reina en conflicte i la posem en una de les files que menys conflictes li causa. Si hi ha empat, triem a l'atzar. Si esgotem el límit de passos, podem provar altres llavors.

In [ ]:
def conflictes(estat, columna, fila):
    return sum(xoquen(i, f, columna, fila)
               for i, f in enumerate(estat) if i != columna)

def min_conflicts(n, rng, passos=1000):
    estat = [rng.randrange(n) for _ in range(n)]
    for pas in range(passos + 1):
        problematiques = [c for c, f in enumerate(estat) if conflictes(estat, c, f)]
        if not problematiques:
            return tuple(estat)
        if pas == passos:
            return None
        columna = rng.choice(problematiques)
        costos = [conflictes(estat, columna, fila) for fila in range(n)]
        millors_files = [fila for fila, cost in enumerate(costos) if cost == min(costos)]
        estat[columna] = rng.choice(millors_files)

solucio = min_conflicts(8, Random(42))
if solucio is not None:
    mostra_tauler(solucio)
else:
    print("Sense solució / Sin solución")